# EPC Ingestion — Bronze → Silver

Reads all London EPC certificate CSVs, filters to social rented, cleans, writes to silver Parquet.

In [3]:
import os
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, trim, when, to_date, year as spark_year
from pyspark.sql.types import FloatType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('epc_ingest') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark ready')

Spark ready


## 1. Read all certificate CSVs into Bronze

In [4]:
RAW_PATH    = '../data/bronze/epc_raw/certificates-*.csv'
SILVER_PATH = '../data/silver/epc'

# Read all years at once — Spark globs the wildcard
raw = spark.read.csv(RAW_PATH, header=True, inferSchema=False)

print(f'Total rows across all years: {raw.count():,}')
print(f'Columns: {len(raw.columns)}')
raw.printSchema()

Total rows across all years: 23,546,857
Columns: 93
root
 |-- certificate_number: string (nullable = true)
 |-- address1: string (nullable = true)
 |-- address2: string (nullable = true)
 |-- address3: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- posttown: string (nullable = true)
 |-- address: string (nullable = true)
 |-- constituency: string (nullable = true)
 |-- constituency_label: string (nullable = true)
 |-- local_authority: string (nullable = true)
 |-- local_authority_label: string (nullable = true)
 |-- built_form: string (nullable = true)
 |-- co2_emiss_curr_per_floor_area: string (nullable = true)
 |-- co2_emissions_current: string (nullable = true)
 |-- co2_emissions_potential: string (nullable = true)
 |-- construction_age_band: string (nullable = true)
 |-- current_energy_efficiency: string (nullable = true)
 |-- current_energy_rating: string (nullable = true)
 |-- energy_consumption_current: string (nullable = true)
 |-- energy_consumption_pote

## 2. Explore tenure values — what does 'social rented' look like?

In [5]:
# See all unique tenure values so we know what to filter on
display(raw.groupBy('tenure').count().orderBy('count', ascending=False).limit(20).toPandas().style.format(thousands=","))

+----------------+-------+
|tenure          |count  |
+----------------+-------+
|owner-occupied  |8863555|
|rented (private)|3788578|
|rented (social) |3467613|
|Owner-occupied  |2594955|
|unknown         |2310577|
|NULL            |990384 |
|Rented (social) |780305 |
|Rented (private)|631066 |
|Unknown         |119819 |
|N/A             |3      |
|N               |2      |
+----------------+-------+



## 3. Filter → Clean → Silver

In [ ]:
silver = (
    raw
    # ── select only the columns we need ──────────────────────────
    .select(
        trim(col('certificate_number')).alias('certificate_id'),
        trim(col('postcode')).alias('postcode'),
        trim(col('local_authority')).alias('local_authority_code'),
        trim(col('local_authority_label')).alias('borough'),
        trim(col('tenure')).alias('tenure'),
        trim(col('property_type')).alias('property_type'),
        trim(col('built_form')).alias('built_form'),
        trim(col('construction_age_band')).alias('construction_age_band'),
        trim(col('current_energy_rating')).alias('epc_rating'),
        col('current_energy_efficiency').cast(FloatType()).alias('epc_score'),
        col('total_floor_area').cast(FloatType()).alias('floor_area_m2'),
        trim(col('main_fuel')).alias('main_fuel'),
        col('co2_emissions_current').cast(FloatType()).alias('co2_emissions'),
        col('energy_consumption_current').cast(FloatType()).alias('energy_consumption'),
        col('heating_cost_current').cast(FloatType()).alias('heating_cost'),
        trim(col('walls_energy_eff')).alias('walls_energy_eff'),
        trim(col('roof_energy_eff')).alias('roof_energy_eff'),
        trim(col('windows_energy_eff')).alias('windows_energy_eff'),
        trim(col('mains_gas_flag')).alias('mains_gas_flag'),
        to_date(col('inspection_date'), 'yyyy-MM-dd').alias('inspection_date'),
        trim(col('region')).alias('region_code'),
    )
    # ── filter to social rented only ─────────────────────────────
    .filter(
        upper(col('tenure')).isin(
            'RENTAL (SOCIAL)',
            'SOCIAL RENTED',
            'RENTED (SOCIAL)'
        )
    )
    # ── filter to London only (region code E12000007) ─────────────
    .filter(col('region_code') == 'E12000007')
    # ── drop rows with no EPC rating ──────────────────────────────
    .filter(col('epc_rating').isNotNull())
    # ── add below_c flag — 2030 government retrofit target ─────────
    .withColumn(
        'below_epc_c',
        when(col('epc_rating').isin('D', 'E', 'F', 'G'), True).otherwise(False)
    )
    # ── add inspection year for trend analysis ────────────────────
    .withColumn('inspection_year', spark_year(col('inspection_date')))
    # ── normalise fuel type — collapses schema-drift duplicates and ──
    # ── distinguishes community heating (building-level retrofit)   ──
    .withColumn(
        'fuel_category',
        when(col('main_fuel').isin(
            'mains gas (not community)', 'Gas: mains gas'
        ), 'gas_individual')
        .when(col('main_fuel') == 'mains gas (community)', 'gas_community')
        .when(col('main_fuel').isin(
            'electricity (not community)', 'Electricity: electricity, unspecified tariff'
        ), 'electric_individual')
        .when(col('main_fuel') == 'electricity (community)', 'electric_community')
        .when(col('main_fuel').isNotNull(), 'other')
        .otherwise(None)
    )
)

print(f'Silver rows (London social rented): {silver.count():,}')
display(silver.limit(5).toPandas().style.format(thousands=","))

## 4. Null audit

In [7]:
from pyspark.sql.functions import count, isnan

def null_count_expr(c, dtype):
    if dtype in ('double', 'float'):
        return count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    return count(when(col(c).isNull(), c)).alias(c)

display(silver.select([null_count_expr(c, t) for c, t in silver.dtypes]).toPandas().style.format(thousands=","))

+--------------+--------+--------------------+-------+------+-------------+----------+---------------------+----------+---------+-------------+---------+-------------+------------------+------------+----------------+---------------+------------------+--------------+---------------+-----------+-----------+---------------+
|certificate_id|postcode|local_authority_code|borough|tenure|property_type|built_form|construction_age_band|epc_rating|epc_score|floor_area_m2|main_fuel|co2_emissions|energy_consumption|heating_cost|walls_energy_eff|roof_energy_eff|windows_energy_eff|mains_gas_flag|inspection_date|region_code|below_epc_c|inspection_year|
+--------------+--------+--------------------+-------+------+-------------+----------+---------------------+----------+---------+-------------+---------+-------------+------------------+------------+----------------+---------------+------------------+--------------+---------------+-----------+-----------+---------------+
|             0|       0|      

## 5. EPC rating distribution — sanity check

In [8]:
display(silver.groupBy('epc_rating').count().orderBy('epc_rating').toPandas().style.format(thousands=","))

# What % is below C?
total = silver.count()
below_c = silver.filter(col('below_epc_c') == True).count()
print(f'Below EPC C: {below_c:,} / {total:,} = {below_c/total*100:.1f}%')

+----------+------+
|epc_rating| count|
+----------+------+
|         A|   323|
|         B| 30227|
|         C|322047|
|         D|211876|
|         E| 42185|
|         F|  4317|
|         G|  1382|
+----------+------+

Below EPC C: 259,760 / 612,357 = 42.4%


## 6. Write to Silver Parquet, partitioned by borough

In [9]:
silver.write \
    .mode('overwrite') \
    .partitionBy('borough') \
    .parquet(SILVER_PATH)

print(f'Silver EPC written to {SILVER_PATH}')

# Verify
verify = spark.read.parquet(SILVER_PATH)
print(f'Verification read: {verify.count():,} rows')

Silver EPC written to ../data/silver/epc
Verification read: 612,357 rows
